In [1]:
import pandas as pd
import time
# np.random.seed(42)
# random.seed(42)

In [2]:
# Load data
# Read the .csv.gz file
df = pd.read_csv('data\soc-sign-bitcoinotc.csv.gz', names=["SOURCE", "TARGET", "RATING", "TIME"])

In [3]:
# Prepare data
# Create a dictionary for neighbors for fast lookups
outgoing_neighbors = df.groupby('SOURCE')['TARGET'].apply(set).to_dict()
incoming_neighbors = df.groupby('TARGET')['SOURCE'].apply(set).to_dict()

# Combine neighbors for undirected-like behavior
all_neighbors = {node: outgoing_neighbors.get(node, set()).union(incoming_neighbors.get(node, set()))
                 for node in set(outgoing_neighbors).union(incoming_neighbors)}

print(f"Total nodes in the graph: {len(all_neighbors)}")

Total nodes in the graph: 5881


In [4]:
def snowball_sampling_pandas(neighbors_dict, seed_node, depth=2):
    """
    Perform snowball sampling using a neighbors dictionary.

    Parameters:
        neighbors_dict (dict): A dictionary where keys are nodes, and values are sets of neighbors.
        seed_node (int/str): The starting node for sampling.
        depth (int): The number of layers to expand.

    Returns:
        set: A set of nodes included in the sampled subgraph.
    """
    sampled_nodes = set([seed_node])  # Start with the seed node
    current_layer = set([seed_node])

    for _ in range(depth):
        next_layer = set()
        for node in current_layer:
            next_layer.update(neighbors_dict.get(node, []))  # Add neighbors of current node
        next_layer -= sampled_nodes  # Avoid revisiting nodes
        sampled_nodes.update(next_layer)
        current_layer = next_layer  # Move to the next layer

    return sampled_nodes

# Start snowball sampling from a seed node
seed = df['SOURCE'].iloc[0]  # Pick the first node as a seed
sampled_nodes = snowball_sampling_pandas(outgoing_neighbors, seed_node=seed, depth=1)

print(f"Sampled {len(sampled_nodes)} nodes.")

Sampled 41 nodes.


In [7]:
# Filter rows where both SOURCE and TARGET are in sampled_nodes
sampled_edges = df[df['SOURCE'].isin(sampled_nodes) & df['TARGET'].isin(sampled_nodes)]

print(f"Sampled subgraph has {len(sampled_edges)} edges.")
print(f"Original graph has {len(df)} edges.")
print(f"ratio: {len(sampled_edges)/len(df)*100}, expected: {len(sampled_nodes)/len(all_neighbors)*100}")

Sampled subgraph has 239 edges.
Original graph has 35592 edges.
ratio: 0.671499213306361, expected: 0.6971603468797823
